# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egeklc/flyrank-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The Random Forest model is trained to predict is_declining, a label indicating whether a page is declining. The model evaluates test pages and ranks them based on their estimated risk of decline. Each page receives a reason code and recommended action to help the team decide which pages to review first and why, while keeping the final decision with human reviewers.

<br>

| Reason code | Condition | Action code |
|---|---|---|
| `stale_visible_low_ctr` | The page has not been updated for at least 91 days, has at least 500 impressions in the last 90 days, and its CTR is below the median CTR for its position tier. | `review_content_and_search_performance` |
| `stale_visible_page` | The page has not been updated for at least 91 days and has at least 500 impressions in the last 90 days. | `review_content_freshness` |
| `low_ctr_for_position` | The page's average position falls within the `top_3`, `page_1`, or `striking` tier, and its CTR is below the median CTR for its position tier. | `review_search_performance` |
| `low_visibility` | The page's average position falls within the `page_3_5` or `deep` tier. | `review_search_visibility` |
| `thin_visible_page` | The page has at least 500 impressions in the last 90 days and its word count is at or below the 25th percentile among pages with at least 500 impressions. | `review_content_depth` |
| `no_strong_signal` | The page does not meet any of the conditions listed above. | `monitor_page` |

In [1]:
!git clone https://github.com/egeklc/flyrank-internship.git

import os
os.chdir("flyrank-internship")
print(os.getcwd())

fatal: destination path 'flyrank-internship' already exists and is not an empty directory.
/content/flyrank-internship


In [2]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

In [3]:
volume_floor = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 500)].copy()
expected_ctr = volume_floor.groupby("position_tier")["ctr"].median().rename("expected_ctr")



df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["stale"] = (df["days_since_last_update"] >= 91).astype(int)

df["expected_ctr"] = (df["position_tier"].map(expected_ctr))
df["ctr_gap"] = (df["expected_ctr"] - df["ctr"])
df["ctr_gap"] = df["ctr_gap"].clip(lower=0)

df.loc[df["avg_position"] <= 0, "ctr_gap"] = 0

word_count_threshold = df["word_count"].quantile(0.25)

In [4]:
def get_reason_action_codes(row):
    if row["stale"] and row["visible"] and row["ctr_gap"] > 0:
        return "stale_visible_low_ctr", "review_content_and_search_performance"

    elif row["stale"] and row["visible"]:
        return "stale_visible_page", "review_content_freshness"

    elif (
        row["ctr_gap"] > 0
        and row["position_tier"] in ["top_3", "page_1", "striking"]
    ):
        return "low_ctr_for_position", "review_search_performance"

    elif row["position_tier"] in ["page_3_5", "deep"]:
        return "low_visibility", "review_search_visibility"

    elif row["visible"] and row["word_count"] <= word_count_threshold:
        return "thin_visible_page", "review_content_depth"

    else:
        return "no_strong_signal", "monitor_page"


In [5]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

target_col = "is_declining"
group_col = "client_id"

excluded_cols = ['content_id', 'client_id',
                 "trend_direction", "trend_pct", "is_declining",
                 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
                 'clicks_prev_30d', 'sessions_prev_30d'
                 ]

num_cols = [col for col in df.select_dtypes(include=["int64", "float64"]).columns
            if col not in excluded_cols
            ]

cat_cols = [col for col in df.select_dtypes(include=["object"]).columns
            if col not in excluded_cols
            ]

feature_cols = num_cols + cat_cols
model_df = df[feature_cols + [target_col, group_col]].copy()


def build_random_forest_model(num_cols, cat_cols):

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0))
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            max_depth=20,
            min_samples_split=15,
            min_samples_leaf=8,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])

    return model

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(scores)[::-1]
    top_k = order[:k]

    return y_true.iloc[top_k].mean()


In [6]:
gss = GroupShuffleSplit(n_splits=1, test_size = 0.25, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df[group_col]))

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

rf = build_random_forest_model(num_cols, cat_cols)

rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

p_at_50 = precision_at_k(y_test, rf_scores, k=50)
rf_auc = roc_auc_score(y_test, rf_scores)

print(f"Precision@50: {p_at_50:.3f}")
print(f"ROC AUC: {rf_auc:.3f}")

Precision@50: 0.640
ROC AUC: 0.603


In [7]:
ranked_test = df.iloc[test_idx].copy()

ranked_test["rf_score"] = rf_scores



ranked_test[["reason_code", "action_code"]] = ranked_test.apply(get_reason_action_codes,
                                                                axis=1,
                                                                result_type="expand")


ranked_test = (ranked_test.sort_values("rf_score", ascending=False).reset_index(drop=True))

ranked_test["rank"] = (ranked_test.index + 1)


action_queue = ranked_test[
    [
        "rank",
        "content_id",
        "rf_score",
        "reason_code",
        "action_code",
    ]
].copy()


top_20_queue = action_queue.head(20)

print("\nTop 20 ranked pages:")
print(top_20_queue.to_string())


Top 20 ranked pages:
    rank            content_id  rf_score            reason_code                            action_code
0      1  content_e988c1699454  0.946102  stale_visible_low_ctr  review_content_and_search_performance
1      2  content_2ba626fea4d6  0.946013   low_ctr_for_position              review_search_performance
2      3  content_9ac61c04930e  0.940389  stale_visible_low_ctr  review_content_and_search_performance
3      4  content_6677fd6c4ea5  0.925170     stale_visible_page               review_content_freshness
4      5  content_98aa0aecb1d9  0.921801  stale_visible_low_ctr  review_content_and_search_performance
5      6  content_f6bf66378677  0.921681   low_ctr_for_position              review_search_performance
6      7  content_35d63627bf3e  0.919067  stale_visible_low_ctr  review_content_and_search_performance
7      8  content_6607847e4454  0.916296  stale_visible_low_ctr  review_content_and_search_performance
8      9  content_694bc660415b  0.915824   low_ctr_

In [8]:
top_50_reason = action_queue[["reason_code"]].head(50).value_counts()
print("\nReason code distribution across the top 50 ranked pages:\n", top_50_reason.to_string())


Reason code distribution across the top 50 ranked pages:
 reason_code          
stale_visible_low_ctr    25
low_ctr_for_position     17
stale_visible_page        8


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** The SEO team uses the ranked queue to prioritize content pages for review and identify potential content or search-performance issues. The reason codes and recommended actions help reviewers understand why a page was flagged and what to investigate.

**Limits:** A high score does not prove that a page will decline. It does not prove that refreshing the content will improve performance or that any specific factor caused the decline. The model ranks pages by their estimated risk of decline based on available data; final decisions require human review.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human review:** Before taking action, the SEO team should check the page's search performance, content quality, freshness, search intent, and relevant context. The reason code is a starting point for investigation, not a definitive diagnosis.

**No-go list:** The system must never automatically refresh, rewrite, remove, or change a page based only on its score. It must not treat a high score as proof of future decline, assume a content refresh will improve performance, or claim that a specific factor caused a decline. All final actions require human review.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Monitor Precision@50 and compare the model's ranked recommendations with actual page performance over time. Reassess or retrain the model if Precision@50 declines, the data distribution changes significantly, or search-performance patterns shift. Review the reason-code rules if they no longer reflect meaningful issues found during human review. Retraining should follow evaluation on newer data, using a validation split that keeps clients separate.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [9]:
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

action_queue.to_csv("work/outputs/action_queue.csv", index=False)
action_queue.to_csv("outputs/action_queue.csv", index=False)
print("Action playbook queue successfully exported to work/outputs/action_queue.csv and outputs/action_queue.csv")

Action playbook queue successfully exported to work/outputs/action_queue.csv and outputs/action_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.